# un_pdfs Track B — fill + validity + dedup (Colab, transformers 4-bit) — **versi cepat**

Jalan paralel dgn Track A (Kaggle). Optimasi kecepatan:
1. **1 model** `Qwen2.5-7B-Instruct` (instruct, tanpa `<think>` bertele-tele) dipakai B1 **dan** B2 → load sekali.
2. B1 `max_new_tokens=1024`.
3. B2 **skip `aimo_hard`** (sudah curated) → langsung ke dedup.

```
after_rules --B1 fill (Qwen)--> filled --B2 validity (Qwen; aimo di-skip)--> valid --B3 dedup--> clean
```
Resumable (cache per `soal` di Drive, lintas-sesi).


## 0. Setup

In [ ]:
!pip install -q transformers accelerate bitsandbytes sentence-transformers faiss-cpu


In [ ]:
from google.colab import drive; drive.mount('/content/drive')
import os, json, gc, torch
from pathlib import Path

REPO_ROOT = Path('/content/drive/MyDrive/DATA_NLP')   # sesuaikan kalau beda
assert (REPO_ROOT / 'data' / 'un_pdfs').exists(), f'cek REPO_ROOT: {REPO_ROOT}'
UN = REPO_ROOT / 'data' / 'un_pdfs'

DATASETS = {'easy': UN / 'easy_after_rules.jsonl', 'aimo_hard': UN / 'aimo_hard_after_rules.jsonl'}
HOLDOUT  = REPO_ROOT / 'data' / 'eval' / 'holdout_v3_un.jsonl'
MODEL    = 'Qwen/Qwen2.5-7B-Instruct'   # satu model utk B1 & B2
LOAD_4BIT= True
FILL_MAX = 1024                          # token max B1 (cukup utk soal SMA)
SKIP_VALIDITY = {'aimo_hard'}            # aimo sudah curated -> lewati B2
def pth(name, stage): return UN / f'{name}_{stage}.jsonl'
for k,v in DATASETS.items(): print(f'{k:10} ada: {v.exists()}')


## 1. Helper (inline)

In [ ]:
import re
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

def read_jsonl(p): return [json.loads(l) for l in open(p, encoding='utf-8') if l.strip()]
def needs_fill(r): return not (r.get('jawaban') or '').strip() or not (r.get('cara') or '').strip()
def strip_think(t):
    i=t.find('</think>'); return t[i+8:].strip() if i>=0 else t.strip()

def extract_boxed(text):
    if not text: return None
    s=text.rfind(r'\boxed')
    if s==-1: return None
    i=s+6
    while i<len(text) and text[i]!='{': i+=1
    if i>=len(text): return None
    depth=0; out=[]
    for ch in text[i:]:
        if ch=='{':
            depth+=1
            if depth==1: continue
        elif ch=='}':
            depth-=1
            if depth==0: return ''.join(out).strip()
        out.append(ch)
    return None

def load_model(model_id):
    tok = AutoTokenizer.from_pretrained(model_id); tok.padding_side='left'
    if tok.pad_token is None: tok.pad_token = tok.eos_token
    kw = dict(torch_dtype=torch.float16, device_map='auto')
    if LOAD_4BIT:
        kw['quantization_config'] = BitsAndBytesConfig(load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16, bnb_4bit_quant_type='nf4')
    return tok, AutoModelForCausalLM.from_pretrained(model_id, **kw).eval()

def free(*objs):
    for o in objs:
        try: del o
        except Exception: pass
    gc.collect(); torch.cuda.empty_cache()

@torch.no_grad()
def gen_batch(tok, model, msgs, max_new_tokens, do_sample, temperature=0.3):
    prompts=[tok.apply_chat_template([{'role':'user','content':m}], tokenize=False, add_generation_prompt=True) for m in msgs]
    enc=tok(prompts, return_tensors='pt', padding=True, truncation=True, max_length=2048).to(model.device)
    out=model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=do_sample,
        temperature=temperature, top_p=0.95, pad_token_id=tok.pad_token_id)
    return tok.batch_decode(out[:, enc['input_ids'].shape[1]:], skip_special_tokens=True)
print('helpers siap')


## 2. Load model (sekali — dipakai B1 & B2)

In [ ]:
tok, model = load_model(MODEL)
print('model loaded:', MODEL)


## B1 — fill GT/cara kosong

In [ ]:
PROMPT=('Selesaikan soal matematika berikut. Tunjukkan langkah-langkah penyelesaian secara rinci '
  r'dalam Bahasa Indonesia. Pastikan jawaban akhir berada di dalam \boxed{{}}.' '\n\n{soal}')
B=8
for name, inp in DATASETS.items():
    if not inp.exists(): print(f'SKIP {name}'); continue
    rows=read_jsonl(inp); cache=pth(name,'fill_cache'); out=pth(name,'filled')
    done={r['soal']: r for r in read_jsonl(cache)} if cache.exists() else {}
    todo=[r for r in rows if needs_fill(r) and r['soal'] not in done]
    print(f'\n== B1 {name} == perlu fill {sum(needs_fill(r) for r in rows)}, sisa {len(todo)}')
    with open(cache,'a',encoding='utf-8') as cf:
        for i in range(0,len(todo),B):
            batch=todo[i:i+B]
            outs=gen_batch(tok, model, [PROMPT.format(soal=r['soal']) for r in batch], FILL_MAX, True)
            for r,t in zip(batch,outs):
                t=strip_think(t); b=extract_boxed(t); jaw=(r.get('jawaban') or '').strip()
                rec={'soal':r['soal'],'cara':t.strip(),'jawaban':jaw or (b or ''),'_ok':bool(jaw) or b is not None}
                done[r['soal']]=rec; cf.write(json.dumps(rec,ensure_ascii=False)+'\n')
            cf.flush(); print(f'   {min(i+B,len(todo))}/{len(todo)}')
    with open(out,'w',encoding='utf-8') as f:
        for r in rows:
            d=done.get(r['soal'])
            f.write(json.dumps({'soal':r['soal'],'cara':(d or r).get('cara',''),'jawaban':(d or r).get('jawaban','')},ensure_ascii=False)+'\n')
    print(f'   filled -> {out}')


## B2 — filter_validity  (aimo_hard di-skip, passthrough)

In [ ]:
VP=('Kamu penilai soal matematika. Tentukan soal berikut VALID atau TIDAK_VALID.\n'
  'VALID jika: lengkap & bisa dipahami dari teks saja; punya jawaban numerik/ekspresi pasti; Bahasa Indonesia jelas.\n'
  'TIDAK_VALID jika: butuh gambar/tabel yang hilang; ambigu; konseptual tanpa jawaban pasti; bukan soal.\n'
  'Jawab HANYA satu kata: VALID atau TIDAK_VALID.\n\nSoal:\n{q}')
def is_valid(t): t=t.strip().upper(); return 'VALID' in t and 'TIDAK_VALID' not in t
B=16
for name in DATASETS:
    fin=pth(name,'filled'); out=pth(name,'valid')
    if not fin.exists(): print(f'SKIP {name}: belum di-fill'); continue
    items=read_jsonl(fin)
    if name in SKIP_VALIDITY:
        with open(out,'w',encoding='utf-8') as f:
            for it in items: f.write(json.dumps(it,ensure_ascii=False)+'\n')
        print(f'== B2 {name} == SKIP (curated) -> passthrough {len(items)} -> {out}'); continue
    prog=pth(name,'valid_prog')
    judged={r['soal']: r['ok'] for r in read_jsonl(prog)} if prog.exists() else {}
    todo=[it for it in items if it['soal'] not in judged]
    print(f'\n== B2 {name} == total {len(items)}, sisa {len(todo)}')
    with open(prog,'a',encoding='utf-8') as pf:
        for i in range(0,len(todo),B):
            batch=todo[i:i+B]
            outs=gen_batch(tok, model, [VP.format(q=it['soal']) for it in batch], 8, False)
            for it,t in zip(batch,outs):
                ok=is_valid(t); judged[it['soal']]=ok
                pf.write(json.dumps({'soal':it['soal'],'ok':ok},ensure_ascii=False)+'\n')
            pf.flush(); print(f'   {min(i+B,len(todo))}/{len(todo)}')
    n=0
    with open(out,'w',encoding='utf-8') as f:
        for it in items:
            if judged.get(it['soal']): f.write(json.dumps(it,ensure_ascii=False)+'\n'); n+=1
    print(f'   {name}: valid {n}/{len(items)} -> {out}')
free(model, tok); print('model dibebaskan (siap B3)')


## B3 — Dedup + dekontaminasi (embedding, CPU)

In [ ]:
import numpy as np, faiss
from sentence_transformers import SentenceTransformer
EMB='sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'; DED=0.92; DEC=0.90
DECONTAM = False   # dekontaminasi vs holdout: matikan sekarang (holdout dibuat baru nanti). Dedup internal TETAP jalan.
emodel=SentenceTransformer(EMB)
def emb(t): return emodel.encode(t, batch_size=128, normalize_embeddings=True, show_progress_bar=True).astype('float32')
bench_emb = emb([(b.get('soal') or b.get('question') or '') for b in read_jsonl(HOLDOUT)]) if (DECONTAM and HOLDOUT.exists()) else None
for name in DATASETS:
    # input B3: pakai hasil B2 (valid) kalau ada; kalau B2 di-skip, fallback ke filled
    fv = pth(name,'valid') if pth(name,'valid').exists() else pth(name,'filled')
    if not fv.exists(): print(f'SKIP {name}'); continue
    items=read_jsonl(fv); E=emb([it['soal'] for it in items])
    idx=faiss.IndexFlatIP(E.shape[1]); keep=[]
    for i in range(len(E)):
        if idx.ntotal>0 and idx.search(E[i:i+1],1)[0][0][0]>=DED: continue
        idx.add(E[i:i+1]); keep.append(i)
    if bench_emb is not None:
        bi=faiss.IndexFlatIP(bench_emb.shape[1]); bi.add(bench_emb)
        keep=[i for i in keep if bi.search(E[i:i+1],1)[0][0][0]<DEC]
    out=pth(name,'clean')
    with open(out,'w',encoding='utf-8') as f:
        for i in keep: f.write(json.dumps(items[i],ensure_ascii=False)+'\n')
    print(f'{name}: {len(items)} -> {len(keep)} -> {out}')


## Cek hasil akhir

In [ ]:
for name in DATASETS:
    f=pth(name,'clean')
    if not f.exists(): print(f'{name}: belum ada clean'); continue
    rows=read_jsonl(f); empt=sum(1 for r in rows if not (r.get('jawaban') or '').strip())
    print(f'{name:10} n={len(rows):5} jawaban_kosong={empt}')
print('\nFile final di Drive: data/un_pdfs/{easy,aimo_hard}_clean.jsonl')
